In [ ]:
# ============================================================
# MESSYMART AI - INITIAL DATA INGESTION
#
# Sources:
#   1. Open Food Facts
#   2. Open Prices
#   3. BigBasketProducts.csv
# ============================================================

import requests
import pandas as pd
import json
import time
from pathlib import Path
from datetime import datetime, timezone

# ============================================================
# CONFIGURATION
# ============================================================

RAW_DIR = Path(r"C:\Users\harsh\OneDrive\Desktop\Dbx project\data\raw")
SOURCE_DIR = Path(r"C:\Users\harsh\OneDrive\Desktop\Dbx project\data\source")

RAW_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

BIGBASKET_FILE = Path(
    r"C:\Users\harsh\OneDrive\Desktop\Dbx project\BigBasketProducts.csv"
)

# Keep this small while testing
OFF_PAGE_SIZE = 20
OFF_MAX_PAGES = 2

OPEN_PRICES_LIMIT = 100

REQUEST_TIMEOUT = 60
MAX_RETRIES = 4

USER_AGENT = (
    "MessyMartAI/1.0 "
    "(data engineering portfolio project)"
)

# ============================================================
# HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "application/json"
})


# ============================================================
# GENERIC GET REQUEST
# ============================================================

def get_with_retry(url, params=None):

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            print(
                f"HTTP {response.status_code} | "
                f"Attempt {attempt}/{MAX_RETRIES}"
            )

            # ------------------------------------------------
            # Temporary server/rate-limit error
            # ------------------------------------------------

            if response.status_code in [429, 500, 502, 503, 504]:

                if attempt < MAX_RETRIES:

                    wait_time = 10 * attempt

                    print(
                        f"Temporary API error "
                        f"({response.status_code})."
                    )

                    print(
                        f"Waiting {wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                    continue

                # Don't crash the whole pipeline
                print(
                    "\nAPI still unavailable "
                    "after retries."
                )

                return None

            response.raise_for_status()

            return response

        except requests.exceptions.RequestException as error:

            print(
                f"Request error: {error}"
            )

            if attempt < MAX_RETRIES:

                wait_time = 10 * attempt

                print(
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                print(
                    "Request failed after all retries."
                )

                return None

    return None


# ============================================================
# 1. OPEN FOOD FACTS
# ============================================================

def fetch_open_food_facts(
    page_size=20,
    max_pages=2
):

    print("\n")
    print("=" * 60)
    print("SOURCE 1: OPEN FOOD FACTS")
    print("=" * 60)

    url = (
        "https://world.openfoodfacts.org/api/v2/search"
    )

    fields = [
        "code",
        "product_name",
        "brands",
        "categories",
        "categories_tags",
        "quantity",
        "packaging",
        "ingredients_text",
        "nutriscore_grade",
        "nutriments",
        "countries",
        "countries_tags",
        "image_url",
        "last_modified_t"
    ]

    all_products = []

    for page in range(1, max_pages + 1):

        print(
            f"\nFetching Open Food Facts page {page}..."
        )

        params = {
            "countries_tags_en": "india",
            "page": page,
            "page_size": page_size,
            "fields": ",".join(fields)
        }

        response = get_with_retry(
            url,
            params=params
        )

        # ----------------------------------------------------
        # If API failed, stop gracefully
        # ----------------------------------------------------

        if response is None:

            print(
                f"\nCould not retrieve page {page}."
            )

            print(
                "Stopping Open Food Facts ingestion "
                "without losing previous pages."
            )

            break

        # ----------------------------------------------------
        # Parse response
        # ----------------------------------------------------

        try:

            data = response.json()

        except ValueError:

            print(
                "Invalid JSON returned by API."
            )

            break

        products = data.get(
            "products",
            []
        )

        if not products:

            print(
                "No products returned."
            )

            break

        all_products.extend(
            products
        )

        print(
            f"Products received: {len(products)}"
        )

        print(
            f"Total collected: {len(all_products)}"
        )

        # ----------------------------------------------------
        # Important: don't hammer the API
        # ----------------------------------------------------

        if page < max_pages:

            print(
                "Waiting before next page..."
            )

            time.sleep(15)

    # ========================================================
    # ADD INGESTION METADATA
    # ========================================================

    ingestion_time = datetime.now(
        timezone.utc
    ).isoformat()

    for product in all_products:

        product["source"] = (
            "open_food_facts"
        )

        product["ingested_at"] = (
            ingestion_time
        )

    # ========================================================
    # SAVE RAW JSON
    # ========================================================

    output_file = (
        RAW_DIR /
        "open_food_facts_india.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            all_products,
            file,
            ensure_ascii=False,
            indent=2
        )

    print("\n")
    print(
        f"Open Food Facts products collected: "
        f"{len(all_products):,}"
    )

    print(
        f"Saved to: {output_file}"
    )

    return all_products


# ============================================================
# 2. OPEN PRICES
# ============================================================

def fetch_open_prices(
    limit=100
):

    print("\n")
    print("=" * 60)
    print("SOURCE 2: OPEN PRICES")
    print("=" * 60)

    url = (
        "https://prices.openfoodfacts.org/"
        "api/v1/prices"
    )

    params = {
        "page_size": limit
    }

    response = get_with_retry(
        url,
        params=params
    )

    # --------------------------------------------------------
    # API unavailable
    # --------------------------------------------------------

    if response is None:

        print(
            "Open Prices could not be retrieved."
        )

        return []

    # --------------------------------------------------------
    # Parse response
    # --------------------------------------------------------

    try:

        data = response.json()

    except ValueError:

        print(
            "Open Prices returned invalid JSON."
        )

        return []

    # --------------------------------------------------------
    # Extract prices
    # --------------------------------------------------------

    if isinstance(data, dict):

        prices = data.get(
            "items",
            data.get(
                "prices",
                []
            )
        )

    elif isinstance(data, list):

        prices = data

    else:

        prices = []

    # ========================================================
    # ADD METADATA
    # ========================================================

    ingestion_time = datetime.now(
        timezone.utc
    ).isoformat()

    for price in prices:

        if isinstance(price, dict):

            price["source"] = (
                "open_prices"
            )

            price["ingested_at"] = (
                ingestion_time
            )

    # ========================================================
    # SAVE
    # ========================================================

    output_file = (
        RAW_DIR /
        "open_prices.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            prices,
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"\nOpen Prices observations: "
        f"{len(prices):,}"
    )

    print(
        f"Saved to: {output_file}"
    )

    return prices


# ============================================================
# 3. BIGBASKET
# ============================================================

def load_bigbasket(
    csv_path
):

    print("\n")
    print("=" * 60)
    print("SOURCE 3: BIGBASKET")
    print("=" * 60)

    csv_path = Path(
        csv_path
    )

    # --------------------------------------------------------
    # Check file
    # --------------------------------------------------------

    if not csv_path.exists():

        print(
            "\nERROR: BigBasket CSV not found."
        )

        print(
            "\nExpected location:"
        )

        print(
            csv_path
        )

        print(
            "\nPut your file here:"
        )

        print(
            r"C:\Users\harsh\OneDrive\Desktop\Dbx project\data\source\BigBasketProducts.csv"
        )

        return None

    # --------------------------------------------------------
    # Load CSV
    # --------------------------------------------------------

    print(
        f"\nReading: {csv_path}"
    )

    df = pd.read_csv(
        csv_path
    )

    # --------------------------------------------------------
    # Add metadata
    # --------------------------------------------------------

    df["source"] = (
        "bigbasket"
    )

    df["ingested_at"] = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )

    # --------------------------------------------------------
    # Save raw copy
    # --------------------------------------------------------

    output_file = (
        RAW_DIR /
        "bigbasket_raw.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

    # --------------------------------------------------------
    # Print information
    # --------------------------------------------------------

    print(
        f"\nRows: {len(df):,}"
    )

    print(
        f"Columns: {len(df.columns)}"
    )

    print(
        "\nColumns:"
    )

    for column in df.columns:

        print(
            f"  - {column}"
        )

    print(
        f"\nSaved to: {output_file}"
    )

    return df


# ============================================================
# 4. INSPECT OPEN FOOD FACTS
# ============================================================

def inspect_open_food_facts(
    products
):

    print("\n")
    print("=" * 60)
    print("OPEN FOOD FACTS INSPECTION")
    print("=" * 60)

    if not products:

        print(
            "No products available."
        )

        return

    df = pd.json_normalize(
        products
    )

    print(
        f"\nRows: {len(df):,}"
    )

    print(
        f"Columns: {len(df.columns)}"
    )

    print(
        "\nAvailable fields:"
    )

    for column in df.columns:

        print(
            f"  - {column}"
        )

    print(
        "\nFirst 5 products:"
    )

    print(
        df.head()
    )


# ============================================================
# 5. INSPECT OPEN PRICES
# ============================================================

def inspect_open_prices(
    prices
):

    print("\n")
    print("=" * 60)
    print("OPEN PRICES INSPECTION")
    print("=" * 60)

    if not prices:

        print(
            "No price observations available."
        )

        return

    df = pd.json_normalize(
        prices
    )

    print(
        f"\nRows: {len(df):,}"
    )

    print(
        f"Columns: {len(df.columns)}"
    )

    print(
        "\nAvailable fields:"
    )

    for column in df.columns:

        print(
            f"  - {column}"
        )

    print(
        "\nFirst 5 price observations:"
    )

    print(
        df.head()
    )


# ============================================================
# 6. MAIN
# ============================================================

def main():

    print("\n")
    print("#" * 60)
    print("# MESSYMART AI")
    print("# INITIAL DATA INGESTION")
    print("#" * 60)

    # --------------------------------------------------------
    # Open Food Facts
    # --------------------------------------------------------

    off_products = fetch_open_food_facts(
        page_size=OFF_PAGE_SIZE,
        max_pages=OFF_MAX_PAGES
    )

    # --------------------------------------------------------
    # Open Prices
    # --------------------------------------------------------

    price_data = fetch_open_prices(
        limit=OPEN_PRICES_LIMIT
    )

    # --------------------------------------------------------
    # BigBasket
    # --------------------------------------------------------

    bigbasket_df = load_bigbasket(
        BIGBASKET_FILE
    )

    # --------------------------------------------------------
    # Inspect
    # --------------------------------------------------------

    inspect_open_food_facts(
        off_products
    )

    inspect_open_prices(
        price_data
    )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print("\n")
    print("#" * 60)
    print("# INGESTION SUMMARY")
    print("#" * 60)
    print(
        f"\nOpen Food Facts: "
        f"{len(off_products):,} products"
    )
    
    print(
        f"Open Prices: "
        f"{len(price_data):,} observations"
    )
    
    if bigbasket_df is not None:
        print(
            f"BigBasket: "
            f"{len(bigbasket_df):,} products"
        )
        
    else:
        print(
            "BigBasket: CSV not loaded"
        )
    print(
        "\nRaw data directory:"
    )

    print(
        RAW_DIR
    )

    print(
        "\nPipeline completed."
    )
# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()



############################################################
# MESSYMART AI
# INITIAL DATA INGESTION
############################################################


SOURCE 1: OPEN FOOD FACTS

Fetching Open Food Facts page 1...
HTTP 200 | Attempt 1/4
Products received: 20
Total collected: 20
Waiting before next page...

Fetching Open Food Facts page 2...
HTTP 503 | Attempt 1/4
Temporary API error (503).
Waiting 10 seconds...
HTTP 503 | Attempt 2/4
Temporary API error (503).
Waiting 20 seconds...
HTTP 503 | Attempt 3/4
Temporary API error (503).
Waiting 30 seconds...
HTTP 503 | Attempt 4/4

API still unavailable after retries.

Could not retrieve page 2.
Stopping Open Food Facts ingestion without losing previous pages.


Open Food Facts products collected: 20
Saved to: C:\Users\harsh\OneDrive\Desktop\Dbx project\data\raw\open_food_facts_india.json


SOURCE 2: OPEN PRICES
HTTP 200 | Attempt 1/4

Open Prices observations: 10
Saved to: C:\Users\harsh\OneDrive\Desktop\Dbx project\data\raw\o

In [16]:
pip install json

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\harsh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
